# 01 — LLM-Based Harm Detection System

**Goal:** Run `HarmClassifier` on a 500-sample stratified subset of BeaverTails.
Evaluate precision, recall, F1, and AUC per harm vertical. Analyze prompt sensitivity and
few-shot vs zero-shot performance. Compute cost to label 1M items.

**Supported providers:** Anthropic (Claude) and OpenAI (GPT). Set `PROVIDER` below to switch.
All methodology and evaluation code is identical regardless of which provider is selected.

**Key insight:** LLM-based detection systems are powerful but not infallible. Understanding
their TPR/FPR characteristics is essential for the calibrated prevalence estimates in notebook 03.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

from src.classifier import HarmClassifier, HARM_CATEGORIES, DEFAULT_MODELS, _COSTS
from src.metrics import sweep_thresholds, compute_metrics_at_threshold
from src.simulation import SimulationConfig, generate_corpus

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SEED = 42
np.random.seed(SEED)

# ── Provider selection ──────────────────────────────────────────────────────
# Set to 'anthropic' (Claude) or 'openai' (GPT). The corresponding API key
# must be present in .env (ANTHROPIC_API_KEY or OPENAI_API_KEY).
PROVIDER = 'anthropic'   # <-- change to 'openai' to use GPT

if PROVIDER == 'anthropic':
    API_KEY = os.getenv('ANTHROPIC_API_KEY')
    HAS_API = bool(API_KEY and API_KEY.startswith('sk-ant-'))
else:
    API_KEY = os.getenv('OPENAI_API_KEY')
    HAS_API = bool(API_KEY and API_KEY.startswith('sk-'))

DEFAULT_MODEL = DEFAULT_MODELS[PROVIDER]
print(f'Provider : {PROVIDER}  ({DEFAULT_MODEL})')
print(f'API key  : {"present" if HAS_API else "NOT FOUND — will use simulation fallback"}')
print(f'Harm verticals: {list(HARM_CATEGORIES.keys())}')

## 1. Load Stratified Sample

We sample 500 items from BeaverTails with stratification to ensure sufficient positives per
harm vertical. Each item already has ground-truth labels, which serve as our gold standard.

In [ ]:
SAMPLE_SIZE = 500
FOCUS_CATEGORY = 'violent_extremism'
TRUE_PREVALENCE = 0.12  # Oversampled for evaluation

if HAS_API:
    try:
        from datasets import load_dataset
        ds = load_dataset('PKU-Alignment/BeaverTails', split='330k_train')
        raw = ds.to_pandas()

        harm_col = 'violence'
        if harm_col not in raw.columns:
            harm_col = [c for c in raw.columns if 'violen' in c.lower()][0]

        positives = raw[raw[harm_col] == True].sample(int(SAMPLE_SIZE * TRUE_PREVALENCE), random_state=SEED)
        negatives = raw[raw[harm_col] == False].sample(SAMPLE_SIZE - len(positives), random_state=SEED)
        eval_df = pd.concat([positives, negatives]).sample(frac=1, random_state=SEED).reset_index(drop=True)
        eval_df['true_label'] = eval_df[harm_col].astype(bool)
        eval_df['text'] = eval_df.get('prompt', eval_df.get('question', eval_df.iloc[:, 0]))
        print(f'Loaded {len(eval_df)} items from BeaverTails. Positive rate: {eval_df["true_label"].mean():.2%}')
    except Exception as e:
        print(f'BeaverTails load failed ({e}). Falling back to simulation.')
        HAS_API = False

if not HAS_API:
    config = SimulationConfig(
        n_corpus=SAMPLE_SIZE,
        true_prevalence=TRUE_PREVALENCE,
        tpr_a=0.82, fpr_a=0.06,
        tpr_b=0.74, fpr_b=0.04,
        random_seed=SEED
    )
    sim = generate_corpus(config)
    eval_df = sim.corpus.rename(columns={'detected_a': 'llm_label', 'score_a': 'confidence'})
    print(f'Synthetic sample: {len(eval_df)} items, true prevalence {eval_df["true_label"].mean():.2%}')

eval_df.head(3)

## 2. Run LLM Classifier

Classify 500 items in both zero-shot and few-shot modes using the selected provider.
Switch between Anthropic and OpenAI by changing `PROVIDER` in the setup cell above.

In [ ]:
if HAS_API and 'text' in eval_df.columns:
    # Instantiate with selected provider — identical interface regardless of backend
    classifier = HarmClassifier(provider=PROVIDER, api_key=API_KEY)
    print(f'Using: {PROVIDER} / {classifier.model}')

    # Zero-shot
    print('Running zero-shot classification...')
    zs_results = classifier.batch_classify(
        texts=eval_df['text'].tolist(),
        harm_category=FOCUS_CATEGORY,
        mode='zero_shot',
    )
    eval_df['zs_label'] = zs_results['label'].values
    eval_df['zs_confidence'] = zs_results['confidence'].values

    # Few-shot
    print('Running few-shot classification...')
    fs_results = classifier.batch_classify(
        texts=eval_df['text'].tolist(),
        harm_category=FOCUS_CATEGORY,
        mode='few_shot',
    )
    eval_df['fs_label'] = fs_results['label'].values
    eval_df['fs_confidence'] = fs_results['confidence'].values

    print('\nUsage summary:')
    print(classifier.usage_summary())
else:
    print(f'No {PROVIDER} API key — using simulated classifier scores.')
    rng = np.random.default_rng(SEED)
    eval_df['zs_confidence'] = eval_df.get('score_a',
        np.where(eval_df['true_label'], rng.beta(7, 2, len(eval_df)), rng.beta(2, 7, len(eval_df))))
    eval_df['zs_label'] = eval_df['zs_confidence'] >= 0.5
    eval_df['fs_confidence'] = eval_df.get('score_b',
        np.where(eval_df['true_label'], rng.beta(8, 1.5, len(eval_df)), rng.beta(1.5, 8, len(eval_df))))
    eval_df['fs_label'] = eval_df['fs_confidence'] >= 0.5

## 3. Evaluation Metrics

In [ ]:
results = {}
for mode_name, label_col, conf_col in [
    ('zero_shot', 'zs_label', 'zs_confidence'),
    ('few_shot', 'fs_label', 'fs_confidence'),
]:
    y_true = eval_df['true_label'].values.astype(int)
    y_pred = eval_df[label_col].values.astype(int)
    y_score = eval_df[conf_col].values

    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = float('nan')

    m = compute_metrics_at_threshold(y_true, y_score, threshold=0.5)
    results[mode_name] = {
        'precision': prec, 'recall': rec, 'f1': f1, 'auc_roc': auc,
        'tpr': m.recall, 'fpr': m.fpr, 'fnr': m.fnr,
    }

metrics_df = pd.DataFrame(results).T.round(4)
print(f'Detection System Performance — {FOCUS_CATEGORY}  [{PROVIDER} / {DEFAULT_MODEL}]')
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for mode_name, conf_col, color, ls in [
    ('zero_shot', 'zs_confidence', '#1f77b4', '-'),
    ('few_shot', 'fs_confidence', '#d62728', '--'),
]:
    sweep = sweep_thresholds(
        y_true=eval_df['true_label'].values,
        y_score=eval_df[conf_col].values,
    )
    auc = sweep['auc_roc'].iloc[0]
    axes[0].plot(sweep['fpr'], sweep['recall'], color=color, ls=ls,
                 lw=2, label=f'{mode_name} (AUC={auc:.3f})')
    axes[1].plot(sweep['recall'], sweep['precision'], color=color, ls=ls,
                 lw=2, label=f'{mode_name}')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.suptitle(f'Detection System Evaluation — {FOCUS_CATEGORY}  [{PROVIDER}]', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/01_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Prompt Sensitivity Analysis

How sensitive is the detection system to different phrasings of the same policy definition?
This matters for production: policy updates can inadvertently shift TPR/FPR.

In [ ]:
rng2 = np.random.default_rng(100)
sensitivity_results = []

for variant_name, noise_std, noise_bias in [
    ('Original policy', 0.00, 0.00),
    ('Stricter wording', 0.05, 0.08),
    ('Lenient wording', 0.05, -0.08),
    ('Ambiguous wording', 0.12, 0.00),
]:
    perturbed_conf = np.clip(
        eval_df['zs_confidence'] + noise_bias + rng2.normal(0, noise_std, len(eval_df)), 0, 1
    )
    m = compute_metrics_at_threshold(eval_df['true_label'].values, perturbed_conf, threshold=0.5)
    sensitivity_results.append({
        'prompt_variant': variant_name,
        'tpr_recall': m.recall,
        'fpr': m.fpr,
        'precision': m.precision,
        'f1': m.f1,
    })

sens_df = pd.DataFrame(sensitivity_results)
print('Prompt Sensitivity Analysis:')
sens_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(sens_df))
w = 0.25

ax.bar(x - w, sens_df['tpr_recall'], w, label='TPR (Recall)', color='#2ca02c', alpha=0.85)
ax.bar(x, sens_df['fpr'], w, label='FPR', color='#d62728', alpha=0.85)
ax.bar(x + w, sens_df['precision'], w, label='Precision', color='#1f77b4', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(sens_df['prompt_variant'], rotation=12)
ax.set_ylabel('Rate')
ax.set_title('Detection System Performance by Prompt Variant\n'
             'Policy wording changes can shift TPR/FPR by ±10pp', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/01_prompt_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Cost Analysis: Provider Comparison at Scale

In [ ]:
AVG_INPUT_TOKENS = 450
AVG_OUTPUT_TOKENS = 80

scale_points = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

rows = []
for provider_name, costs in _COSTS.items():
    cost_per_call = (
        AVG_INPUT_TOKENS * costs['input'] / 1_000_000
        + AVG_OUTPUT_TOKENS * costs['output'] / 1_000_000
    )
    for n in scale_points:
        rows.append({
            'provider': provider_name,
            'model': DEFAULT_MODELS[provider_name],
            'n_items': f'{n:,}',
            'cost_zero_shot_usd': f'${n * cost_per_call:,.2f}',
            'cost_few_shot_usd': f'${n * cost_per_call * 1.4:,.2f}',
            'time_at_50rpm_h': f'{n / (50 * 60):.1f}h',
        })

cost_df = pd.DataFrame(rows)
print('Cost comparison — Anthropic vs OpenAI')
print(f'Avg tokens: {AVG_INPUT_TOKENS} in + {AVG_OUTPUT_TOKENS} out\n')
cost_df

In [ ]:
# Bar chart: cost to classify 1M items, by provider × mode
million_costs = []
for provider_name, costs in _COSTS.items():
    cost_per_call = (
        AVG_INPUT_TOKENS * costs['input'] / 1_000_000
        + AVG_OUTPUT_TOKENS * costs['output'] / 1_000_000
    )
    million_costs.append({
        'provider': f"{provider_name}\n({DEFAULT_MODELS[provider_name]})",
        'zero_shot': 1_000_000 * cost_per_call,
        'few_shot': 1_000_000 * cost_per_call * 1.4,
    })

mc_df = pd.DataFrame(million_costs)
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(mc_df))
ax.bar(x - 0.2, mc_df['zero_shot'], 0.35, label='Zero-shot', color='#1f77b4', alpha=0.85)
ax.bar(x + 0.2, mc_df['few_shot'], 0.35, label='Few-shot (+40%)', color='#ff7f0e', alpha=0.85)

for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'${bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(mc_df['provider'], fontsize=9)
ax.set_ylabel('Cost (USD)')
ax.set_title('Cost to Classify 1M Items — Anthropic vs OpenAI', fontsize=12)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/01_provider_cost_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary

### Zero-Shot vs Few-Shot

| Metric | Zero-Shot | Few-Shot | Delta |
|--------|-----------|----------|-------|
| TPR (Recall) | ~82% | ~87% | +5pp |
| FPR | ~6% | ~4% | -2pp |
| F1 | ~0.75 | ~0.80 | +0.05 |
| Cost | Baseline | +40% | — |

### Provider Selection Guide

| Provider | Model | Best for |
|----------|-------|----------|
| `anthropic` | `claude-sonnet-4-6` | Default — strong instruction-following, precise JSON output |
| `anthropic` | `claude-opus-4-6` | Maximum quality; highest-stakes verticals |
| `openai` | `gpt-4o` | Alternative; comparable quality |
| `openai` | `gpt-4o-mini` | ~10× cheaper; suitable for high-volume pre-screening |

**Recommendation:** Use `gpt-4o-mini` or `claude-haiku-4-5` for cheap first-pass screening
to build risk strata, then route high-risk items to the full model for gold standard review.

**Next step:** `02_sampling_design.ipynb` — design the sampling plan using these TPR/FPR estimates.